In [4]:
import torch

resnet50 = torch.hub.load('facebookresearch/dino:main', 'dino_resnet50')

Using cache found in /home/rtxmsi1/.cache/torch/hub/facebookresearch_dino_main


In [5]:
for name, module in resnet50.named_modules():
    print(f"{name}")


conv1
bn1
relu
maxpool
layer1
layer1.0
layer1.0.conv1
layer1.0.bn1
layer1.0.conv2
layer1.0.bn2
layer1.0.conv3
layer1.0.bn3
layer1.0.relu
layer1.0.downsample
layer1.0.downsample.0
layer1.0.downsample.1
layer1.1
layer1.1.conv1
layer1.1.bn1
layer1.1.conv2
layer1.1.bn2
layer1.1.conv3
layer1.1.bn3
layer1.1.relu
layer1.2
layer1.2.conv1
layer1.2.bn1
layer1.2.conv2
layer1.2.bn2
layer1.2.conv3
layer1.2.bn3
layer1.2.relu
layer2
layer2.0
layer2.0.conv1
layer2.0.bn1
layer2.0.conv2
layer2.0.bn2
layer2.0.conv3
layer2.0.bn3
layer2.0.relu
layer2.0.downsample
layer2.0.downsample.0
layer2.0.downsample.1
layer2.1
layer2.1.conv1
layer2.1.bn1
layer2.1.conv2
layer2.1.bn2
layer2.1.conv3
layer2.1.bn3
layer2.1.relu
layer2.2
layer2.2.conv1
layer2.2.bn1
layer2.2.conv2
layer2.2.bn2
layer2.2.conv3
layer2.2.bn3
layer2.2.relu
layer2.3
layer2.3.conv1
layer2.3.bn1
layer2.3.conv2
layer2.3.bn2
layer2.3.conv3
layer2.3.bn3
layer2.3.relu
layer3
layer3.0
layer3.0.conv1
layer3.0.bn1
layer3.0.conv2
layer3.0.bn2
layer3.0.conv

In [12]:
dict([*resnet50.named_children()])

{'conv1': Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False),
 'bn1': BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
 'relu': ReLU(inplace=True),
 'maxpool': MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False),
 'layer1': Sequential(
   (0): Bottleneck(
     (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
     (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
     (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
     (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (relu): ReLU(inplace=True)
     (downsample): Sequential(
       (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
   

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

class PerceptualLoss(nn.Module):
    def __init__(self, layers=None, resize=True):
        super(PerceptualLoss, self).__init__()
        #self.model = torch.hub.load('facebookresearch/dino:main', 'dino_resnet50')
        self.model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')

        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False

        # Layers to extract features from
        # relu — after the initial conv layer (captures low-level edges/textures).
        # layer1.0.relu — early block (low-mid level features).
        # layer2.0.relu — mid-level features.
        # layer3.0.relu — higher-level features.
        # layer4.0.relu — highest-level semantic features.

        self.layers = layers or ['relu', 'layer1', 'layer2', 'layer3', 'layer4']
        self.resize = resize

        # Hook the layers
        self.feature_maps = {}
        for layer_name in self.layers:
            layer = dict([*self.model.named_children()])[layer_name]
            layer.register_forward_hook(self._get_hook(layer_name))

        # Normalization transform (ImageNet stats)
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                              std=[0.229, 0.224, 0.225])

    def _get_hook(self, name):
        def hook(module, input, output):
            self.feature_maps[name] = output
        return hook

    def forward(self, input_img, target_img):
        if self.resize:
            input_img = F.interpolate(input_img, size=(224, 224), mode='bilinear', align_corners=False)
            target_img = F.interpolate(target_img, size=(224, 224), mode='bilinear', align_corners=False)

        # Normalize images
        input_img = self.normalize(input_img)
        target_img = self.normalize(target_img)

        # Clear stored features
        self.feature_maps = {}

        # Forward pass through the network
        _ = self.model(input_img)
        input_features = self.feature_maps.copy()

        self.feature_maps = {}
        _ = self.model(target_img)
        target_features = self.feature_maps.copy()

        # Compute perceptual loss
        loss = 0
        for layer in self.layers:
            feat_input = input_features[layer]
            feat_target = target_features[layer]
            loss += F.mse_loss(feat_input, feat_target, reduction="sum")

        return loss


In [21]:
import torch
from torch import nn

# Paste the full PerceptualLoss class definition here (from the previous message)
# ...

# Step 1: Instantiate the loss function
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loss_fn = PerceptualLoss(layers=[
    'relu', 
    'layer1', 
    'layer2', 
    'layer3', 
    'layer4'
]).to(device)

# Step 2: Create dummy input tensors (batch size 2, 3 channels, 256x256)
dummy_input = torch.rand(2, 3, 256, 256).to(device)
dummy_target = torch.rand(2, 3, 256, 256).to(device)

# Step 3: Compute the perceptual loss
with torch.no_grad():  # no need for gradients in this dummy example
    loss = loss_fn(dummy_input, dummy_target)

print(f"Perceptual Loss: {loss.item():.4f}")


Using cache found in /home/rtxmsi1/.cache/torch/hub/facebookresearch_dino_main


Perceptual Loss: 24152368.0000


In [23]:
dinov2_vits14 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')

Using cache found in /home/rtxmsi1/.cache/torch/hub/facebookresearch_dinov2_main
/home/rtxmsi1/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/rtxmsi1/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/rtxmsi1/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /home/rtxmsi1/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth
100%|██████████| 84.2M/84.2M [00:28<00:00, 3.11MB/s]


In [24]:
for name, module in dinov2_vits14.named_modules():
    print(f"{name}")


patch_embed
patch_embed.proj
patch_embed.norm
blocks
blocks.0
blocks.0.norm1
blocks.0.attn
blocks.0.attn.qkv
blocks.0.attn.attn_drop
blocks.0.attn.proj
blocks.0.attn.proj_drop
blocks.0.ls1
blocks.0.drop_path1
blocks.0.norm2
blocks.0.mlp
blocks.0.mlp.fc1
blocks.0.mlp.act
blocks.0.mlp.fc2
blocks.0.mlp.drop
blocks.0.ls2
blocks.0.drop_path2
blocks.1
blocks.1.norm1
blocks.1.attn
blocks.1.attn.qkv
blocks.1.attn.attn_drop
blocks.1.attn.proj
blocks.1.attn.proj_drop
blocks.1.ls1
blocks.1.drop_path1
blocks.1.norm2
blocks.1.mlp
blocks.1.mlp.fc1
blocks.1.mlp.act
blocks.1.mlp.fc2
blocks.1.mlp.drop
blocks.1.ls2
blocks.1.drop_path2
blocks.2
blocks.2.norm1
blocks.2.attn
blocks.2.attn.qkv
blocks.2.attn.attn_drop
blocks.2.attn.proj
blocks.2.attn.proj_drop
blocks.2.ls1
blocks.2.drop_path1
blocks.2.norm2
blocks.2.mlp
blocks.2.mlp.fc1
blocks.2.mlp.act
blocks.2.mlp.fc2
blocks.2.mlp.drop
blocks.2.ls2
blocks.2.drop_path2
blocks.3
blocks.3.norm1
blocks.3.attn
blocks.3.attn.qkv
blocks.3.attn.attn_drop
blocks.3

In [1]:
from torchvision import models
convnext = models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1).eval()
convnext

Downloading: "https://download.pytorch.org/models/convnext_small-0c510722.pth" to /home/rtxmsi1/.cache/torch/hub/checkpoints/convnext_small-0c510722.pth
100%|██████████| 192M/192M [00:14<00:00, 13.9MB/s] 


ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=

In [2]:
import torch
from torchvision import models
import torchvision.transforms.functional as TF

# Load pretrained ConvNeXt model
convnext = models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1).eval()

# ImageNet normalization values
_IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

# Create dummy input and target tensors (batch of 1, 3 channels, 256x256)
input = torch.rand(1, 3, 256, 256)
target = torch.rand(1, 3, 256, 256)

# Resize to 224x224
input = torch.nn.functional.interpolate(input, size=224, mode="bilinear", align_corners=False, antialias=True)
target = torch.nn.functional.interpolate(target, size=224, mode="bilinear", align_corners=False, antialias=True)

# Normalize
input_norm = (input - _IMAGENET_MEAN) / _IMAGENET_STD
target_norm = (target - _IMAGENET_MEAN) / _IMAGENET_STD

# Forward pass
with torch.no_grad():
    pred_input = convnext(input_norm)
    pred_target = convnext(target_norm)

# Output the shapes
print("Input prediction shape:", pred_input.shape)
print("Target prediction shape:", pred_target.shape)


Input prediction shape: torch.Size([1, 1000])
Target prediction shape: torch.Size([1, 1000])
